# Project 2 — Image Classification with a Neural Network 🔢
### Teaching a network to read handwritten digits (your first real step toward OCR)

In Project 1 you learned the **7 universal steps** of an ML project on simple tables. Now we apply the *exact same 7 steps* to **images** — but the model becomes a **neural network**, and the phrase *"training is just adjusting weights to reduce error"* becomes a loop **you write and watch run**.

**What we're building:** a model that looks at a 28×28 image of a handwritten digit and says which digit (0–9) it is. That is literally *baby OCR* — recognising characters from pixels. Project 3 (full OCR) just extends this to whole lines of text.

---
### How this project works (the new methodology in action)
- 🔁 = a core concept carried over from Project 1 (the repetition is on purpose).
- ✍️ = **a cell where YOU write the code** (scaffold + hints given; solution in the next cell — try first).
- 🐛 = a **planted bug** for you to diagnose and fix.
- 🔮 = **predict the output before running** — then check yourself.

**Compute note:** this trains fine on a normal CPU in a few minutes (we keep it small). The code auto-uses a GPU if you have one. If your machine is slow, lower `EPOCHS` or use Google Colab's free GPU.


## Step 0 — Setup

New tool: **PyTorch** (`torch`), the deep-learning library. Two ideas to meet right away:
- **Tensor** — just a multi-dimensional array (like a NumPy array) that can also run on a GPU and remember how to compute gradients. An image is a tensor; a batch of images is a bigger tensor.
- **device** — *where* the math happens: `"cuda"` (GPU, fast) or `"cpu"`. We write device-agnostic code so it runs anywhere.


In [6]:
%pip install torchvision

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 1.9 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.1 MB 2.1 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.1 MB 2.0 MB/s eta 0:00:02
   ----------------- ---------------------- 1.8/4.1 MB 2.1 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.1 MB 2.1 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.1 MB 2.2 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.1 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 2.3 MB/s  0:00:01
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/123.0 MB 2.8 MB/s eta 0:00:45
   ---------------------------

  You can safely remove it manually.


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)   # reproducibility 🔁 (same idea as random_state in Project 1)
print("PyTorch", torch.__version__, "| device:", device)

RuntimeError: operator torchvision::nms does not exist

## Step 1 — Frame the problem 🔁

Same three questions as always:

**Q1. What are we predicting?** Which digit (0–9) a handwritten image shows.

**Q2. What type of problem?** Predicting a category → **classification**. Specifically **multi-class** (10 possible classes).
> 🔁 Compare your past work: loan default was *binary* classification (2 classes: yes/no). This is the same idea scaled to 10 classes. House prices (Project 1) was *regression* (a number). Same family, different flavour.

**Q3. Inputs and output?**
- **Features (X)** = the 28×28 = 784 pixel values of the image. Each pixel's brightness is a clue.
- **Target (y)** = the digit label, 0–9.

> 🧠 *The leap from Project 1:* there, features were tidy columns (income, rooms). Here the features are raw pixels with **spatial structure** — pixel (5,5) is next to pixel (5,6), and that *arrangement* is what makes a "7" a "7". Keeping that structure is exactly why we'll need a special kind of network (a CNN), not just stacked columns.


## Step 2 — Get & understand the data (EDA) 🔁

We use **MNIST**: 70,000 grayscale images of handwritten digits — the "hello world" of computer vision. `torchvision` downloads it automatically the first time.

EDA for images = *actually look at them*, check how many there are, and confirm the classes are balanced.


In [ ]:
# transforms.ToTensor() converts a 0-255 image into a 0.0-1.0 tensor (more on this in Step 4)
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=transforms.ToTensor())
test_set   = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())

print("Training images:", len(train_full))
print("Test images:", len(test_set))
img, label = train_full[0]
print("One image is a tensor of shape:", tuple(img.shape), "= [channels, height, width]")
print("Its label is:", label)

In [ ]:
# Look at the data: plot the first 8 digits with their labels
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    image, lbl = train_full[i]
    ax.imshow(image.squeeze(), cmap="gray")  # squeeze() drops the channel dim for plotting
    ax.set_title(f"label: {lbl}"); ax.axis("off")
plt.show()

In [ ]:
# 🔁 Check class balance (in the loan project, imbalance was a big deal!).
import collections
counts = collections.Counter(int(lbl) for _, lbl in train_full)
print("Images per digit:", dict(sorted(counts.items())))
print("Roughly balanced -> plain accuracy will be a fair metric here (unlike imbalanced loan data).")

> 🔁 **Why we still check balance even though we expect it:** in your loan project, imbalance meant accuracy lied (a model predicting 'no default' for everyone looked 95% accurate while being useless). MNIST is balanced, so accuracy is trustworthy here — but you only *know* that because you checked. Checking is the habit; the result varies.


## Step 3 — Split the data: train / validation / test 🔁🔁

Project 1 used train + test. Now we add a third pile: a **validation set**.

- **Training set** — the model learns from these.
- **Validation set** — checked *during* training to watch for overfitting and tune choices, *without* touching the test set.
- **Test set** — the sealed final exam, opened once at the very end.

> 🧠 *Analogy:* training = doing homework; validation = weekly practice quizzes that tell you how you're doing and let you adjust your study; the test set = the final exam you only sit once. If you "studied to" the final exam, your grade would be a lie — that's the leakage idea 🔁 from Project 1, now with three piles instead of two.

MNIST already separates test for us. We carve a validation slice out of the training data.


In [ ]:
val_size = 10_000
train_size = len(train_full) - val_size
train_set, val_set = random_split(train_full, [train_size, val_size])
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

### DataLoaders and batches — why we don't feed all 50,000 images at once

A **DataLoader** serves the data in small **batches** (e.g. 64 images at a time).

**Why batches?** Two reasons:
1. **Memory** — you can't always fit the whole dataset on the GPU at once.
2. **Better learning** — updating the model after each small batch gives many small, slightly-noisy correction steps, which actually helps it learn faster and generalise better than one giant step.

> 🧠 *Analogy:* studying 50,000 flashcards by reviewing them in stacks of 64 and adjusting after each stack — not by trying to memorise all 50,000 in a single glance.


In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)   # shuffle so order isn't memorised
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)

images, labels = next(iter(train_loader))
print("One batch of images:", tuple(images.shape), "= [batch, channels, height, width]")
print("One batch of labels:", tuple(labels.shape))

## Step 4 — Preprocess: normalise the pixels 🔁

Raw pixels run 0–255 (or 0.0–1.0 after `ToTensor`). We **normalise** them to be centred around 0 with a consistent spread.

> 🔁 This is the **exact same idea as feature scaling in Project 1** (`StandardScaler`). Neural networks train faster and more stably when inputs are small and centred — big raw values can make the training steps lurch around. Different data type (pixels vs columns), identical principle.

The standard MNIST normalisation uses its known mean (0.1307) and standard deviation (0.3081). We bake it into the transform.


In [ ]:
normalise = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),   # (mean,), (std,)  -> centres pixels near 0
])
# Re-load with normalisation applied (kept separate above so the EDA plots stayed human-readable)
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=normalise)
test_set   = datasets.MNIST(root="./data", train=False, download=True, transform=normalise)
train_set, val_set = random_split(train_full, [train_size, val_size])
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)
print("Data re-loaded with normalisation. Ready to model.")

## Step 5 — Build the model: a Convolutional Neural Network (CNN) 🆕

### First, what *is* a neural network?
A stack of **layers** made of **neurons**. Each neuron takes numbers in, multiplies them by **weights** (the things that get learned), adds them up, and passes the result on. Stack enough layers and the network can learn very complex input→output relationships.
> 🧠 *Analogy:* an assembly line. Each station (layer) transforms the product a little, passing it to the next, until the final station outputs the answer. "Learning" = adjusting what each station does (the weights).

### Why a *convolutional* network for images?
You *could* flatten the 784 pixels into a row and use plain layers — but that throws away the spatial arrangement (which pixels are next to which). A **convolution** instead slides a small window (a **filter**) across the image looking for a local pattern — an edge, a curve, a corner.

> 🧠 *Analogy:* a small stamp you slide across the whole picture, lighting up wherever it finds its pattern (say, a diagonal stroke). Early filters find edges; later ones combine edges into shapes like loops and lines — the building blocks of digits. This is why CNNs dominate image tasks.

### The layers we'll use
- **Conv2d** — the sliding pattern-detectors (produces "feature maps").
- **ReLU** — an **activation function**: it keeps positive values and zeros out negatives. *Why?* It adds **non-linearity**. Without it, stacking layers just collapses back into one straight line (🔁 like Project 1's linear model) and the network couldn't learn curves. ReLU is what lets depth actually buy you power.
- **MaxPool2d** — **downsampling**: keep the strongest signal in each little 2×2 region, shrinking the image and keeping what matters. Makes the model faster and a bit position-tolerant.
- **Flatten → Linear** — finally squash the feature maps into a row and map to **10 outputs** (one score per digit). These raw scores are called **logits**.


### ✍️ Your turn — complete the forward pass
The layers are defined for you in `__init__`. Fill in `forward()`, which describes how data flows through them. Hints are in the comments. (Solution follows — attempt first.)

> 💡 **Debugging trick you'll reuse forever:** unsure what shape a layer outputs? *Don't do the math in your head — print it.* The `_flatten_size` helper below shows how to discover `32*7*7` by pushing a dummy image through the conv stack. When a shape error bites you, this trick is your fastest fix.


In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1 channel in (grayscale) -> 16 feature maps
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 -> 32 feature maps
        self.pool  = nn.MaxPool2d(2, 2)                            # halves height & width
        self.fc1   = nn.Linear(32 * 7 * 7, 128)                   # 32 maps of 7x7 -> 128
        self.fc2   = nn.Linear(128, num_classes)                  # 128 -> 10 digit scores (logits)

    def forward(self, x):
        # x comes in as [batch, 1, 28, 28]
        # 1. conv1 -> ReLU -> pool   (result: [batch, 16, 14, 14])
        # 2. conv2 -> ReLU -> pool   (result: [batch, 32, 7, 7])
        # 3. flatten to [batch, 32*7*7], then fc1 -> ReLU, then fc2
        # YOUR CODE HERE (replace the line below):
        raise NotImplementedError("Write the forward pass, then delete this line")


In [ ]:
# SOLUTION — compare after attempting
class DigitCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # -> [batch, 16, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))   # -> [batch, 32, 7, 7]
        x = torch.flatten(x, 1)                # -> [batch, 1568]
        x = F.relu(self.fc1(x))
        return self.fc2(x)                      # raw logits, no softmax here (see Step 6)

model = DigitCNN().to(device)

# The print-the-shape debugging trick in action:
dummy = torch.zeros(1, 1, 28, 28).to(device)
with torch.no_grad():
    feat = model.pool(F.relu(model.conv2(model.pool(F.relu(model.conv1(dummy))))))
print("Discovered conv-stack output shape:", tuple(feat.shape), "-> flatten =", feat.numel())
print("That 1568 is exactly what fc1 expects. If it didn't match, THAT is your bug.")

## Step 6 — Train the model 🔁 (the heart of deep learning)

Now the payoff from Project 1. Two ingredients:

**The loss function** — how wrong the model is, as one number.
> 🔁 Project 1 used **MSE** because we predicted a *number*. For *classification* we use **Cross-Entropy Loss**, which measures how far the predicted class probabilities are from the truth. Same role (a "wrongness score" to minimise), chosen to match the task.
> ⚠️ Note: PyTorch's `CrossEntropyLoss` expects **raw logits** and applies softmax internally — so we deliberately did **not** put a softmax in the model. (Adding one here is a classic silent bug.)

**The optimizer** — the rule for adjusting weights to reduce the loss. We use **Adam**, a smart version of gradient descent.
> 🧠 *Analogy, extended from Project 1:* the loss is the radio static; the optimizer turns the dial to reduce it; the **learning rate** is *how far* you turn the dial each step — too big and you overshoot the station, too small and it takes forever.

**Epochs** — one **epoch** = one full pass over all the training data. We do a few.

### The training loop has the same 5 steps every single time
1. `optimizer.zero_grad()` — clear last step's gradients (🐛 the bug below forgets this).
2. forward pass — `outputs = model(images)`.
3. compute loss — `loss = criterion(outputs, labels)`.
4. `loss.backward()` — **backpropagation**: compute how each weight affected the loss.
5. `optimizer.step()` — nudge every weight in the direction that lowers loss.

Memorise these five. Every PyTorch model you ever train uses them.


### ✍️ Your turn — write the training loop body
Fill in the five steps. This is the single most important code to be able to write from memory. Solution follows.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

def train_one_epoch(model, loader):
    model.train()                       # put model in training mode
    running = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        # --- the 5 steps (YOUR CODE HERE) ---
        # 1. zero the gradients
        # 2. forward pass: outputs = ...
        # 3. loss = criterion(...)
        # 4. backward
        # 5. optimizer step
        raise NotImplementedError("Write the 5 steps, then delete this line")
        running += loss.item()
    return running / len(loader)


In [ ]:
# SOLUTION
def train_one_epoch(model, loader):
    model.train()
    running = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                 # 1
        outputs = model(images)               # 2
        loss = criterion(outputs, labels)     # 3
        loss.backward()                       # 4
        optimizer.step()                      # 5
        running += loss.item()
    return running / len(loader)

@torch.no_grad()                              # no gradients needed when just measuring
def eval_loss(model, loader):
    model.eval()
    running = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        running += criterion(model(images), labels).item()
    return running / len(loader)

# 🔮 PREDICT before running: will training loss go DOWN each epoch? Will val loss track it?
train_hist, val_hist = [], []
for epoch in range(EPOCHS):
    tr = train_one_epoch(model, train_loader)
    va = eval_loss(model, val_loader)
    train_hist.append(tr); val_hist.append(va)
    print(f"Epoch {epoch+1}/{EPOCHS}  train loss {tr:.4f} | val loss {va:.4f}")

## Step 7 — Evaluate 🔁

### Accuracy on the sealed test set
Two PyTorch habits when *measuring* (not training):
- `model.eval()` — switches the model to evaluation mode.
- `torch.no_grad()` — skips gradient bookkeeping (faster, and signals "I'm only looking, not learning").

> 🔁 To turn the 10 logits into a prediction, we take the **argmax** — the digit with the highest score. (Softmax would convert logits to probabilities, but for picking the winner, argmax of the logits is enough.)

### ✍️ Your turn — write the accuracy function


In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # YOUR CODE HERE:
        # preds = the predicted digit for each image (hint: outputs.argmax(dim=1))
        # add the number of correct predictions to `correct`, and batch size to `total`
        raise NotImplementedError("Compute preds and tally correct/total, then delete this line")
    return correct / total


In [ ]:
# SOLUTION
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total

print(f"Test accuracy: {accuracy(model, test_loader)*100:.2f}%")

### Diagnose overfitting vs underfitting — now you can *see* it 🔁🔁
In Project 1 we compared a single train vs test number. With epochs, we can **plot the curves** and watch:
- both losses falling and close together → healthy learning.
- train loss falling while **val loss starts rising** → **overfitting** (memorising the training set). 🧠 *the student acing homework but flunking quizzes.*
- both losses stuck high → **underfitting** (model too simple / trained too little).


In [ ]:
plt.plot(range(1, EPOCHS+1), train_hist, marker="o", label="train loss")
plt.plot(range(1, EPOCHS+1), val_hist,   marker="o", label="val loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Watch for overfitting"); plt.legend(); plt.show()
print("If the two lines diverge (train down, val up), that gap is overfitting — same concept as Project 1.")

In [ ]:
# A confusion matrix shows WHICH digits get mistaken for which (richer than one accuracy number) 🔁
from sklearn.metrics import confusion_matrix
import numpy as np
all_preds, all_true = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        all_preds.extend(model(images.to(device)).argmax(1).cpu().numpy())
        all_true.extend(labels.numpy())
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Confusion matrix")
plt.xticks(range(10)); plt.yticks(range(10)); plt.show()
print("Bright off-diagonal cells = common mix-ups (e.g. 4 vs 9, 3 vs 5). This tells you WHERE the model struggles.")

## 🐛 Break-it-and-fix-it: the most famous PyTorch training bug
Below is a training loop with **one line removed**. It will **not crash** — it'll just train badly (loss behaves erratically and won't settle sensibly), because each step's gradients **pile on top of the previous step's** instead of starting fresh.

**Your job:** run it, notice the loss misbehaving, identify the missing line from the 5 steps, and add it back.


In [ ]:
# 🐛 BUGGY — run it, watch the loss act strangely, then fix it
buggy_model = DigitCNN().to(device)
buggy_opt = torch.optim.Adam(buggy_model.parameters(), lr=1e-3)
buggy_model.train()
for i, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    # >>> one of the 5 steps is missing here <<<
    outputs = buggy_model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    buggy_opt.step()
    if i % 100 == 0: print(f"step {i}  loss {loss.item():.3f}")
    if i == 300: break


**Diagnosis:** the missing line is `buggy_opt.zero_grad()` at the top of the loop. Without it, gradients **accumulate** across steps — the optimizer effectively takes wrong, ever-growing steps, so training is unreliable. 🔁 This is the *"the code ran but isn't correct"* lesson from Project 1, in deep-learning form. **Fix:** add `buggy_opt.zero_grad()` as the first line inside the loop.


## Step 8 — Refactor to production + save the model 🏭

Loose training cells are fine for learning, but to **deploy** you need clean functions and a way to **persist** the trained model so you don't retrain every time.

`torch.save` writes the model's learned weights to a file (`.pth`). Loading it back is how a real app serves predictions instantly. *Saving the trained weights is a genuine deployment step.*


In [ ]:
def build_model() -> nn.Module:
    """Create a fresh DigitCNN on the correct device."""
    return DigitCNN().to(device)

def train_model(model, train_loader, val_loader, epochs=3, lr=1e-3):
    """Train and return (model, history). Encapsulates the loop so callers can't get it wrong."""
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    for ep in range(epochs):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = crit(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item()
        history["train"].append(running / len(train_loader))
        history["val"].append(eval_loss(model, val_loader))
        print(f"epoch {ep+1}: train {history['train'][-1]:.4f} | val {history['val'][-1]:.4f}")
    return model, history

def save_model(model, path="digit_cnn.pth"):
    """Persist learned weights so the model can be reloaded without retraining (deployment step)."""
    torch.save(model.state_dict(), path)
    print("Saved ->", path)

def load_model(path="digit_cnn.pth") -> nn.Module:
    """Rebuild the architecture and load saved weights into it."""
    m = build_model()
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

# Demonstrate the clean pipeline + persistence:
save_model(model)                       # save the model you trained above
served = load_model()                   # reload as a deployed app would
print(f"Reloaded model test accuracy: {accuracy(served, test_loader)*100:.2f}%")

In [ ]:
# A "deployment" helper: predict a single image the way an app endpoint would
@torch.no_grad()
def predict_digit(model, image_tensor) -> int:
    """Take one [1,28,28] image tensor and return the predicted digit 0-9."""
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)   # add the batch dimension -> [1,1,28,28]
    return int(model(image_tensor).argmax(1).item())

sample_img, sample_lbl = test_set[7]
print(f"Model predicts: {predict_digit(served, sample_img)}  | actual: {sample_lbl}")

## 🎓 Recap — what you drilled

**Carried over from Project 1 (🔁):** the 7-step workflow; framing (classification vs regression); EDA and checking class balance; the train/test split + leakage, now extended to **train/val/test**; preprocessing as scaling/normalisation; **loss as a wrongness-score to minimise**; **overfitting vs underfitting** diagnosed by the train-vs-val gap; "the code ran ≠ the code is correct."

**New this project:**
- **Neural networks** — layers of weighted neurons; learning = adjusting weights.
- **CNNs** — convolutions as sliding feature-detectors that respect image structure; ReLU for non-linearity; pooling for downsampling; logits at the end.
- **The training loop's 5 steps** — zero_grad → forward → loss → backward → step — which you can now write from memory.
- **Optimizer, learning rate, epochs, batches** — the mechanics of gradient descent.
- **Cross-entropy** for classification; **argmax** for predictions; **confusion matrix** for *where* it fails.
- **Saving/loading a model** — real model persistence for deployment.

### 🔁 The bridge to Project 3 (OCR)
You just built a model that reads *one* character from an image. **OCR** is the same idea extended: locate characters in a larger image and read a *sequence* of them into text. Your digit classifier is the seed of that — we'll grow it next.

### ✍️ Experiments to cement it
1. Train for `EPOCHS = 8` instead of 3. Do the loss curves start to diverge (overfitting)? 
2. Add a `nn.Dropout(0.25)` layer before `fc2` and retrain — does the train/val gap shrink? (Dropout fights overfitting; you'll meet it again.)
3. **Deliberately break it:** add `x = F.softmax(x, dim=1)` to the end of `forward()` and retrain. Accuracy gets worse — why? (Hint: re-read the ⚠️ note in Step 6 about what `CrossEntropyLoss` expects.)

When you've run it through and tried at least one experiment, tell me how the loss curves looked and we'll move to **Project 3 — OCR**.
